In [ ]:
import pandas as pd

roll_number = "1024170020"

fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

last_two_digits = [int(roll_number[-2]), int(roll_number[-1])]
categories = ["billing", "account", "general"]

personal_entry_1 = {
    "question": "how can i update my email address",
    "answer": "Go to Account Settings and update your registered email address.",
    "keywords": "email update account",
    "category": categories[last_two_digits[0] % 3]
}

personal_entry_2 = {
    "question": "how can i check my fee payment status",
    "answer": "Open Billing and check the status of your latest fee payment.",
    "keywords": "fee payment status",
    "category": categories[last_two_digits[1] % 3]
}

all_entries = fixed_entries + [personal_entry_1, personal_entry_2]
df = pd.DataFrame(all_entries)

print("Final 6-row FAQ DataFrame:")
print(df)

In [ ]:
def score_query(query, df):
    query_words = set(query.lower().split())
    scores = []

    for index, row in df.iterrows():
        faq_words = set(
            (row["question"] + " " + row["keywords"]).lower().split()
        )
        score = len(query_words.intersection(faq_words))
        scores.append(score)

    result = df.copy()
    result["confidence"] = scores
    result = result[result["confidence"] > 0]
    result = result.sort_values(by="confidence", ascending=False)

    return result.reset_index(drop=True)

query = "fee payment"
result = score_query(query, df)

print("Q2: Matching FAQs:")
print(result)

In [ ]:
def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()]

category_name = personal_entry_1["category"]

print("Q3: FAQs in category:", category_name)
print(same_category(category_name, df))

In [ ]:
entry_index = 0

new_keyword = input("Q4: Enter a new keyword: ")

df.loc[entry_index, "keywords"] = (
    df.loc[entry_index, "keywords"] + " " + new_keyword
)

file_name = roll_number + "_faq_data.csv"
df.to_csv(file_name, index=False)

print("Updated FAQ entry:")
print(df.loc[entry_index])
print("DataFrame saved as:", file_name)

In [ ]:
print("Q5: Number of FAQ entries per category:")

category_count = df.groupby("category").size()

print(category_count)

In [ ]:
def score_query_with_tie(query, df):
    query_words = set(query.lower().split())
    scores = []

    for index, row in df.iterrows():
        faq_words = set(
            (row["question"] + " " + row["keywords"]).lower().split()
        )
        score = len(query_words.intersection(faq_words))
        scores.append(score)

    result = df.copy()
    result["confidence"] = scores
    result = result[result["confidence"] > 0]
    result = result.sort_values(by="confidence", ascending=False).reset_index(drop=True)

    if len(result) == 0:
        print("No matching FAQ found.")
        return result

    highest_score = result["confidence"].max()
    top_matches = result[result["confidence"] == highest_score]

    print("Highest confidence score:", highest_score)

    if len(top_matches) > 1:
        print("Tie detected! All matching entries are:")
    else:
        print("No tie. Best matching entry is:")

    print(top_matches)
    return top_matches

print("Q6: Tie demonstration")
score_query_with_tie("fee", df)

print("Q6: No-tie demonstration")
score_query_with_tie("password", df)